# Mini Project 2 - Part 1 (100 pts)

## Goal of the Assignment
The goal of this assignment is to answer user queries by retrieving relevant information from a referenced book.

To do this,we will break it down into several steps that include: the design of a Python script that loads a PDF document, breaks it down into smaller chunks, generates embeddings for each chunk using OpenAI's API, stores these embeddings in a Pinecone index, and then uses these embeddings to perform a similarity search and retrieve relevant documents based on a query.

### [Reference Paper on Retrieval Augmented Generation](https://arxiv.org/abs/2005.11401)

## Exercise 0:
1. Go through the OpenAI documnentation here for text embeddings [OpenAPI Docs](https://platform.openai.com/docs/guides/embeddings)
1. Go through the Pinecone and Langchain Integration documentation here [Pinecone](https://docs.pinecone.io/docs/openai)

In [1]:
!pip install langchain
!pip install unstructured
!pip install pdf2image
!pip install pdfminer.six
!pip install unstructured_inference
!pip install pikepdf
!pip install pypdf
!pip install pinecone
!pip install openai
!pip install tiktoken
!pip install pymupdf
!pip install langchain_openai
!pip install langchain-community
!pip install langchain-pinecone

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.2/220.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 36.2 MB/s eta 0:00:00
   

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 57.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.4/53.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 134.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 736.8/736.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompatible.
fastai 2.8.6 requires torch<2.10,>=1.10, but you have torch 2.10.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 15.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.9
    Uninstalling langchain-core-1.2.9:
      Successfully uninstalled langchain-core-1.2.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 29.1 MB/s eta 0:00:00
  Attempting uninstall: pinecone-plugin-assistant
    Found existing installation: pinecone-plugin-assistant 3.0.2
    Uninstalling pinecone-plugin-assistant-3.0.2:
      Successfully 

In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Pinecone
from pinecone import Pinecone, ServerlessSpec
from tqdm.notebook import tqdm
import langchain
import openai
from openai import OpenAI
import string

## Task 1. Load PDF file and extract text (25 pts)

### You are required to load the PDF file and extract the text from it. You can use PyMuPDFLoader to extract text and page numbers from the PDF file. The extracted text and page numbers should be stored in a variables `page_texts` and `page_numbers`.

### Use the provided PDF file 'machine-learning.pdf' to extract the text from it.
### (5 pts)

In [3]:
# from langchain.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import PyMuPDFLoader

# Extract document per page
#############################
## TODO: Write your code here
#############################
# Initialize the loader with the file path
loader = PyMuPDFLoader("./machine-learning.pdf")

# Load the documents (this returns a list of Document objects)
documents = loader.load()

# Print to verify loading
print(f"Loaded {len(documents)} pages.")

# TODO: Extract text content and page numbers from the documents

# Use list comprehension to extract the 'page_content' from each document
page_texts = [doc.page_content for doc in documents]  # Extract page_content

# Use list comprehension to extract the 'page' number from metadata
page_numbers = [doc.metadata["page"] for doc in documents]  # Extract metadata["page"]

Loaded 227 pages.


### Break down the extracted text into smaller text (typical chunk size is around the length of a page) chunks using RecursiveCharacterTextSplitter (20 pts)

1. Initialize chunking parameters (chunk_size=2500, overlap=50) As an intial starting point
2. Create storage lists for chunks and page numbers
3. Loop through each page, extracting text and page numbers
4. Append the previous page’s tail to the current page
5. Apply text chunking with overlap
6. Store chunks and assign correct page numbers
7. Update previous_page_tail to carry over text to the next page

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

#############################
## TODO: Write your code here
#############################
chunk_size, overlap = 2500, 50
#  Initialize the Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=overlap,
    length_function=len,
    is_separator_regex=False
)

# Prepare Variables
chunked_texts, chunk_page_numbers = [], []
previous_page_tail = ""  # Stores the last `overlap` characters from the previous page

# Process Each Page with Cross-Page Overlap

# Loop through each page, process text, and ensure overlap is preserved across pages.
for text, page_number in zip(page_texts, page_numbers):
    # Prepend the previous page’s overlap to the current page:
    current_text_with_overlap = previous_page_tail + text
    # Split text into chunks using RecursiveCharacterTextSplitter:
    chunks = text_splitter.split_text(current_text_with_overlap)
    # Store Chunked Text and Page Numbers
    for chunk in chunks:
      chunked_texts.append(chunk)
      chunk_page_numbers.append(page_number)

    # Update the last `overlap` characters for the next page
    if len(text) > overlap:
      previous_page_tail = text[-overlap:]
    else:
      # Handle edge case where a page has very little text
      previous_page_tail = text

# Verify the result
print(f"Total chunks created: {len(chunked_texts)}")
print(f"Sample chunk: {chunked_texts[0][:100]}...")

Total chunks created: 331
Sample chunk: A Course in
Machine Learning
Hal Daumé III...


## Task 2. Prepare the data (20 pts)
1. Convert the list of texts and page numbers into a DataFrame with a column name 'text' and 'page_number' (5 pts)
2. Preprocess the texts by removing punctuation and new lines (5 pts)
3. Generate embeddings for each text using the embeddings function. (5 pts)
4. Create a new column in the dataframe to store the generated embedding (5 pts)


In [6]:
import pandas as pd
from google.colab import userdata

# Setup OpenAI Client using Secret Key
# Retrieve the API key securely from Colab Secrets
openai_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=openai_key)

# Function to get the embeddings of the text using OpenAI text-embedding-3-small model
def get_embedding(text, model="text-embedding-3-small"):
   text = text.replace("\n", " ")
   return client.embeddings.create(input = [text], model=model).data[0].embedding

In [7]:
#############################
## TODO: Write your code here
#############################

# Convert the list of texts into a DataFrame
df = pd.DataFrame({'text': chunked_texts, 'page_number': chunk_page_numbers})

# Preprocess the texts by removing punctuation and new lines
def clean_text(text):
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Remove new lines
    text = text.replace("\n", " ")
    return text

# Apply the cleaning function to the 'text' column
df['text'] = df['text'].apply(clean_text)

# Verify cleaning
print("Cleaned text example:", df['text'].iloc[0][:100])

# Generate embeddings for each text using the embeddings function
# Store the generated embedding into the dataframe with a column name 'embeddings'
print("Generating embeddings... please wait.")
# Apply the get_embedding function to the 'text' column
# Use a lambda function to pass each text row to the API
df['embeddings'] = df['text'].apply(lambda x: get_embedding(x, model='text-embedding-3-small'))

# Check the result
print(f"Embeddings generated. DataFrame shape: {df.shape}")
print(df.head())

Cleaned text example: A Course in Machine Learning Hal Daumé III
Generating embeddings... please wait.
Embeddings generated. DataFrame shape: (331, 3)
                                                text  page_number  \
0         A Course in Machine Learning Hal Daumé III            0   
1  A Course in Machine Learning Hal Daumé IIICopy...            1   
2  imlinfo TODO   Second printing January 2017For...            2   
3  For my students and teachers Often the sameTAB...            3   
4  Probabilistic Modeling 116 10 Neural Networks ...            4   

                                          embeddings  
0  [0.005700266920030117, 0.02941744774580002, 0....  
1  [0.00669109309092164, 0.026317643001675606, 0....  
2  [-0.019943011924624443, 0.008549213409423828, ...  
3  [0.016484133899211884, 0.026472298428416252, 0...  
4  [-0.004062092397361994, -0.0393034890294075, 0...  


## Task 3. Create Pinecone index and insert the data (15 pts)
1. Create a Pinecone index on [`Pinecone Console`](https://www.pinecone.io/) with OpenAI text embedding size dimensions and cosine similarity metric (5 pts)
2. Initialize Pinecone client and connect to the Pinecone index
3. Insert the embeddings, text and other appropriate meta data(in a dictionary see below for example) into the Pinecone index (5 pts). Create namespaces according to the chunking, say you are using chunks of 500, 1000 use namespaces ns500,ns-1000 respectively
4. Get the index info and print the number of records in the index (5 pts)


#### Refer [`Pinecone docs`](https://docs.pinecone.io/docs/overview)

## Format for tuple required for Pinecone index
```python
(
    "unique_id_for_each_record",
    document_embedding_vector,
    {
        "text": "document_text",
        "num_tokens": "number_of_tokens_in_document"
        "other_meta_data": "other_meta_data"
    }
)
```

Example upsert into `pinecone_index`
```python
pinecone_index.upsert(
  vectors=[
    {"id": "vec1", "values": [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], "metadata": {"text": [....],"page_number": [1]}},
    {"id": "vec2", "values": [0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2], "metadata": {"text": [....],"page_number": [1]}},
    {"id": "vec3", "values": [0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3], "metadata": {"text": [....],"page_number": [2]}},
    {"id": "vec4", "values": [0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4], "metadata": {"text": [....],"page_number": [3]}}
  ],
  namespace="ns500"
)

pinecone_index.upsert(
  vectors=[
    {"id": "vec5", "values": [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5], "metadata": {"text": [....],"page_number": [5]}},
    {"id": "vec6", "values": [0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6], "metadata": {"text": [....],"page_number": [5]}},
    {"id": "vec7", "values": [0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7], "metadata": {"text": [....],"page_number": [6]}},
    {"id": "vec8", "values": [0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8], "metadata": {"text": [....],"page_number": [7]}}
  ],
  namespace="ns1000"
)
```

In [9]:

from pinecone import Pinecone, ServerlessSpec
import os, time

# ---- 3.2 Initialize Pinecone client and connect to the Pinecone index ----
# Get API key from Colab Secrets
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
INDEX_NAME = "ml-mp2"        # TODO: change to your Pinecone index name

pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index
existing = [x["name"] for x in pc.list_indexes()]
if INDEX_NAME not in existing:
    print(f"Creating index: {INDEX_NAME}...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    # wait until ready
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(2)
    print("Index created successfully!")
else:
    print(f"Index {INDEX_NAME} already exists.")

# Connect to the index
index = pc.Index(INDEX_NAME)
print("Connected to:", INDEX_NAME)


#############################
# TODO: Write your code here (3.3)
#############################
# ---- 3.3 Insert (upsert) embeddings + text + metadata ----
# Insert the embeddings, text and other appropriate meta data
# Format: (id, vector, metadata)
vectors_to_upsert = []

# Define namespace based on your chunk size
namespace_name = "ns2500"

print("Preparing vectors for upload...")

# Iterate through the DataFrame rows
for i, row in df.iterrows():
    # Create a unique ID for each chunk (e.g., "vec_0", "vec_1")
    vec_id = f"vec_{i}"

    # Get the embedding vector
    vec_values = row['embeddings']

    # Create metadata dictionary
    vec_metadata = {
        "text": row['text'],
        "page_number": int(row['page_number']) # Ensure it's a standard Python int
    }

    # Append to the list
    vectors_to_upsert.append({
        "id": vec_id,
        "values": vec_values,
        "metadata": vec_metadata
    })

# Upsert to Pinecone
# Using batching is good practice to avoid hitting size limits
batch_size = 100
print(f"Uploading {len(vectors_to_upsert)} vectors to namespace '{namespace_name}'...")

for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i : i + batch_size]
    index.upsert(vectors=batch, namespace=namespace_name)

print("Upload complete!")

Creating index: ml-mp2...
Index created successfully!
Connected to: ml-mp2
Preparing vectors for upload...
Uploading 331 vectors to namespace 'ns2500'...
Upload complete!


In [10]:
# 3.4 Printout the index stats
print("Index stats:", index.describe_index_stats())

Index stats: {'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'ns2500': {'vector_count': 331}},
 'total_vector_count': 331,
 'vector_type': 'dense'}


In [11]:
# Upload different chunk sizes
def process_and_upload(target_chunk_size, target_namespace):
    print(f"\n--- Processing for Chunk Size {target_chunk_size} (Namespace: {target_namespace}) ---")

    # 1. Chunking
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=target_chunk_size,
        chunk_overlap=50, # Keep overlap consistent
        length_function=len,
        is_separator_regex=False
    )

    local_chunks = []
    local_pages = []
    prev_tail = ""

    # Reuse page_texts and page_numbers from Task 1
    for text, page_num in zip(page_texts, page_numbers):
        text_with_overlap = prev_tail + text
        chunks = splitter.split_text(text_with_overlap)
        for c in chunks:
            local_chunks.append(c)
            local_pages.append(page_num)
        # Update tail
        prev_tail = text[-50:] if len(text) > 50 else text

    # 2. DataFrame & Cleaning
    temp_df = pd.DataFrame({'text': local_chunks, 'page_number': local_pages})
    temp_df['text'] = temp_df['text'].apply(clean_text) # Reuse your clean_text function

    print(f"Generated {len(temp_df)} chunks. Generating embeddings...")

    # 3. Embeddings
    # Reuse your get_embedding function
    temp_df['embeddings'] = temp_df['text'].apply(lambda x: get_embedding(x))

    # 4. Upsert to Pinecone
    vectors = []
    for i, row in temp_df.iterrows():
        vectors.append({
            "id": f"vec_{i}",
            "values": row['embeddings'],
            "metadata": {"text": row['text'], "page_number": int(row['page_number'])}
        })

    # Batch upsert
    batch_size = 100
    for i in range(0, len(vectors), batch_size):
        index.upsert(vectors=vectors[i:i+batch_size], namespace=target_namespace)

    print(f"Successfully uploaded to {target_namespace}!")

process_and_upload(1000, "ns1000")
process_and_upload(500, "ns500")


--- Processing for Chunk Size 1000 (Namespace: ns1000) ---
Generated 662 chunks. Generating embeddings...
Successfully uploaded to ns1000!

--- Processing for Chunk Size 500 (Namespace: ns500) ---
Generated 1261 chunks. Generating embeddings...
Successfully uploaded to ns500!


## Task 4. Query the vector store Implementation (30 pts)
1. Initialize the vectorstore with the Pinecone index and the embeddings (refer Pinecone docs). (5 pts)
2. Create a function to perform a similarity search on the vectorstore with a query and return the most relevant documents top-5 (top-k). (10 pts)
3. Call the query_pinecone_vector_store function and print the text and page numbers for the top 5 queries (5 pts)
3. Experiment with different text chunk sizes(min 3) for querying the Pinecone index and report the best chunk size. Also explain the reason for the best chunk size (10 pts)

In [14]:
#############################
## TODO: Create Pinecone vector store
#############################
from langchain_pinecone import PineconeVectorStore

# Initialize embeddings object (needed to convert query to vector)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=openai_key)

# text_key="text" tells LangChain that the content is stored in the 'text' metadata field
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    text_key="text"
)

In [15]:
#############################
## TODO: Function to query the Pinecone vector store and return the top-k results

def query_pinecone_vector_store(query: str, top_k: int = 5, nameSpace: str = "ns500"):
    """
    Searches the Pinecone index for the most similar documents to the query.

    Args:
        query: The user's question.
        top_k: Number of results to return.
        nameSpace: The specific namespace to search in (e.g., 'ns500', 'ns1000', 'ns2500').

    Returns:
        List of Document objects found.
    """
    # Perform similarity search filtering by namespace
    results = vectorstore.similarity_search(
        query,
        k=top_k,
        namespace=nameSpace
    )
    return results


In [16]:
#############################
## TODO: Call query_pinecone_vector_store and print the text and the Page number for your query
############################
# Test Query
test_query = "What is the definition of machine learning?"

print(f"--- Querying Namespace: ns2500 with query: '{test_query}' ---")
results = query_pinecone_vector_store(test_query, top_k=3, nameSpace="ns2500")

# Print results
for i, doc in enumerate(results):
    print(f"\nResult {i+1} (Page {doc.metadata['page_number']}):")
    print(f"Content snippet: {doc.page_content[:200]}...") # Print first 200 chars
    print("-" * 50)

--- Querying Namespace: ns2500 with query: 'What is the definition of machine learning?' ---

Result 1 (Page 7.0):
Content snippet: check github for the latest list of contributors1  DECISION TREES Dependencies None At a basic level machine learning is about predicting the future based on the past For instance you might wish to pr...
--------------------------------------------------

Result 2 (Page 9.0):
Content snippet: nduce a function f This function f will be evalu10 a course in machine learning ated on the test data The machine learning algorithm has succeeded if its performance on the test data is high 12 Some C...
--------------------------------------------------

Result 3 (Page 26.0):
Content snippet: job of the development data is to allow us to tunelimits of learning 27 hyperparameters The general approach is as follows 1 Split your data into 70 training data 10 development data and 20 test data ...
--------------------------------------------------


In [17]:
##############################
## TODO: Experiment with different text chunk sizes(min 3) for querying the Pinecone index and report the best chunk size.
#############################
queries = ["What is supervised learning?", "Explain the bias-variance tradeoff."]
namespaces = ["ns500", "ns1000", "ns2500"]

for ns in namespaces:
    print(f"\n\n====== Testing Namespace: {ns} ======")
    for q in queries:
        print(f"\nQuery: {q}")
        res = query_pinecone_vector_store(q, top_k=1, nameSpace=ns)
        if res:
            print(f"Top Result (Page {res[0].metadata['page_number']}):")
            print(f"Text: {res[0].page_content[:150]}...")
        else:
            print("No results found.")



====== Testing Namespace: ns500 ======

Query: What is supervised learning?
Top Result (Page 177.0):
Text: 00001 145 Further Reading TODO further reading15  UNSUPERVISED LEARNING Dependencies If you have access to labeled training data you know what to do T...

Query: Explain the bias-variance tradeoff.
Top Result (Page 70.0):
Text: might be data saturated 59 BiasVariance Tradeoff Because one of the key questions in machine learning is the question of representation it is common t...


====== Testing Namespace: ns1000 ======

Query: What is supervised learning?
Top Result (Page 35.0):
Text: What is the difference between un supervised and supervised learning that means that we know what the “right answer” is for supervised learning but no...

Query: Explain the bias-variance tradeoff.
Top Result (Page 70.0):
Text: near 0 error then you need to work on better feature design or pick another learning model eg decision tree versus linear model If not you probably do...


====== Testing N

In [ ]:
#############################
## TODO: explain the reason for the best chunk size
############################

### Analysis of Chunk Sizes

After experimenting with chunk sizes of 500, 1000, and 2500 characters, we determined that **1000 characters** is the optimal size for this textbook.

**Reasoning:**

1.  **Context vs. Fragmentation (vs. 500)**:
    * The **500-character** chunks were sometimes too fragmented. For example, when querying *"What is supervised learning?"*, the top result came from the *Unsupervised Learning* chapter (Page 177). While it contained keywords, it lacked the proper introductory context found by the larger chunks.
    * The **1000-character** chunk (ns1000) correctly retrieved the direct definition and comparison on Page 35 ("What is the difference between unsupervised and supervised learning..."), providing a much more accurate answer.

2.  **Noise Reduction (vs. 2500)**:
    * The **2500-character** chunks were too large and introduced "noise" into the embeddings. For the same query, the top result (Page 73) focused on specific examples (true/false questions) rather than the fundamental definition. The relevant information was likely diluted by the large amount of surrounding text, lowering the retrieval quality.

**Conclusion:**
A chunk size of **1000** offers the best balance. It is large enough to capture complete semantic concepts (like definitions) but small enough to maintain a focused vector representation, ensuring the retrieval system finds the most precise answers.

## Task 5. Retrieval Augmented Generation (10 pts)
Once you have identified the most similar text to your query from the Pinecone database, you will use this text as an input prompt to a language model. Specifically, you will be using the GPT-3.5 model provided by the OpenAI API. The model will generate a query answer based on the input prompt. (10 pts)

The prompt should handle only the queries that are only relevant to the document.
Ex: If my query is "How to cook an egg?"
The response should be " this query is not relevant to the context of this book. I would be happy to answer the question based on the books context."

In [18]:
#############################
## TODO: Using OpenAI API answer the query
## TODO: with the top-k results as context
#############################
def get_rag_response(query: str, namespace: str = "ns1000"):
    """
    Retrieves context from Pinecone and generates an answer using GPT-3.5.
    If the query is irrelevant, returns a standard refusal message.
    """

    # 1. Retrieve top-k context chunks from Pinecone
    # We use the function defined in Task 4.
    # Using 'ns1000' as it was determined to be the best chunk size.
    retrieved_docs = query_pinecone_vector_store(query, top_k=5, nameSpace=namespace)

    # 2. Construct the Context String
    # Combine the text from the retrieved documents
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # 3. Define the Prompt for GPT
    # The system prompt enforces the strict rule about irrelevant queries.
    system_prompt = """You are a helpful teaching assistant for a Machine Learning textbook.

    Instructions:
    1. Answer the user's Question strictly using ONLY the provided Context.
    2. If the answer cannot be found in the Context, or if the Question is not relevant to the domain of Machine Learning or the provided text, you MUST respond with EXACTLY this sentence:
    "this query is not relevant to the context of this book. I would be happy to answer the question based on the books context."
    3. Do not invent information not present in the Context.
    """

    user_prompt = f"Context:\n{context_text}\n\nQuestion:\n{query}"

    # 4. Call OpenAI API (GPT-3.5-turbo)
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0 # Set to 0 to minimize hallucinations
    )

    return response.choices[0].message.content


In [19]:
## TODO: Create 5 test prompts to get a response from GPT. Make sure to include some relevant and some irrelevant
# Define a list of test queries (Mixed relevant and irrelevant)
test_queries = [
    "What is supervised learning?",        # Relevant
    "How to cook an egg?",            # Irrelevant (Completely off-topic)
    "Explain the bias-variance tradeoff.",    # Relevant
    "Who won the Super Bowl in 2026?",     # Irrelevant (Fact not in book)
    "What is a decision tree?"          # Relevant
]

print("------ RAG System Test Results ------\n")

# Loop through queries and print responses
for q in test_queries:
    print(f"User Query: {q}")
    ans = get_rag_response(q)
    print(f"AI Response: {ans}")
    print("-" * 50)

------ RAG System Test Results ------

User Query: What is supervised learning?
AI Response: "Supervised learning is the setting in which you have a teacher telling you the right answers."
--------------------------------------------------
User Query: How to cook an egg?
AI Response: this query is not relevant to the context of this book. I would be happy to answer the question based on the book's context.
--------------------------------------------------
User Query: Explain the bias-variance tradeoff.
AI Response: "this query is not relevant to the context of this book. I would be happy to answer the question based on the book's context."
--------------------------------------------------
User Query: Who won the Super Bowl in 2026?
AI Response: this query is not relevant to the context of this book. I would be happy to answer the question based on the book's context.
--------------------------------------------------
User Query: What is a decision tree?
AI Response: A decision tree i